## 单隐藏层多层感知机（MLP）

### 网络结构

```
输入层 (d)  →  隐藏层 (256)  →  输出层 (C)
   x       W1,b1     h      W2,b2     y
  (d,)     →      ReLU →     →     (C,)
```

- **输入层**: $d$ 个神经元（特征维度）
- **隐藏层**: **256** 个神经元，后接 ReLU 激活函数
- **输出层**: $C$ 个神经元（类别数）

### 数学表达

$$h = \text{ReLU}(W_1 x + b_1)$$
$$y = W_2 h + b_2$$

其中 $W_1 \in \mathbb{R}^{256 \times d}$，$W_2 \in \mathbb{R}^{C \times 256}$。


In [1]:
import torch
from torch import nn

# ========== 方式一：使用 nn.Module 子类 ==========
class MLP(nn.Module):
    """多层感知机 — 单隐藏层 (256 个神经元)"""
    def __init__(self, input_dim, hidden_dim=256, output_dim=10):
        super().__init__()
        self.hidden = nn.Linear(input_dim, hidden_dim)   # 输入 → 隐藏层 (256)
        self.relu = nn.ReLU()                            # 非线性激活
        self.output = nn.Linear(hidden_dim, output_dim)  # 隐藏层 → 输出
    
    def forward(self, x):
        x = self.hidden(x)       # 线性变换: (N, d) → (N, 256)
        x = self.relu(x)         # 激活: 引入非线性
        x = self.output(x)       # 线性变换: (N, 256) → (N, C)
        return x

# ========== 方式二：使用 nn.Sequential（更简洁） ==========
def create_mlp(input_dim, hidden_dim=256, output_dim=10):
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim),   # (N, d) → (N, 256)
        nn.ReLU(),                          # 激活
        nn.Linear(hidden_dim, output_dim)   # (N, 256) → (N, C)
    )

if __name__ == "__main__":
    # 示例：28×28 图片 → 10 分类
    batch_size, input_dim, output_dim = 4, 784, 10
    
    # 实例化
    model = MLP(input_dim=input_dim, hidden_dim=256, output_dim=output_dim)
    print("=" * 50)
    print("模型结构:")
    print(model)
    
    # 前向传播
    x = torch.randn(batch_size, input_dim)
    out = model(x)
    print(f"\n输入形状: {x.shape}")     # (4, 784)
    print(f"输出形状: {out.shape}")    # (4, 10)
    
    # 参数量统计
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n总参数量:     {total:,}")
    print(f"可训练参数量: {trainable:,}")
    
    # 各层参数明细
    print("\n各层参数:")
    for name, param in model.named_parameters():
        print(f"  {name:20s}  形状: {str(param.shape):15s}  参数: {param.numel():,}")

模型结构:
MLP(
  (hidden): Linear(in_features=784, out_features=256, bias=True)
  (relu): ReLU()
  (output): Linear(in_features=256, out_features=10, bias=True)
)

输入形状: torch.Size([4, 784])
输出形状: torch.Size([4, 10])

总参数量:     203,530
可训练参数量: 203,530

各层参数:
  hidden.weight         形状: torch.Size([256, 784])  参数: 200,704
  hidden.bias           形状: torch.Size([256])  参数: 256
  output.weight         形状: torch.Size([10, 256])  参数: 2,560
  output.bias           形状: torch.Size([10])  参数: 10
